# Hybrid Results Presentation

This notebook is the compact paper-facing figure and table notebook for the current Bologna grouped multisensor result.


In [ ]:
%matplotlib widget
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
from IPython.display import display, Image as IPyImage

ROOT = Path('/home/ubuntu/work/insar_mcmc')

OVERLAP_SUMMARY = ROOT / 'outputs_bologna_2025_overlap' / 'bologna_mintpy2025_w3ra_overlap_summary.json'
TILE_SUMMARY = ROOT / 'outputs_stage1_bologna_multisensor_kalman_tiled_overlap2025_smaprefresh' / 'stage1_bologna_multisensor_kalman_tiled_summary.json'
TILE_RESULTS = ROOT / 'outputs_stage1_bologna_multisensor_kalman_tiled_overlap2025_smaprefresh' / 'stage1_bologna_multisensor_kalman_tiled_results.npz'
REGIONAL_SUMMARY = ROOT / 'outputs_stage1_bologna_multisensor_kalman_overlap2025_smaprefresh' / 'stage1_bologna_multisensor_kalman_summary.json'
REGIONAL_RESULTS = ROOT / 'outputs_stage1_bologna_multisensor_kalman_overlap2025_smaprefresh' / 'stage1_bologna_multisensor_kalman_results.npz'
SYNTH_SUMMARY = ROOT / 'outputs_stage1_pure' / 'stage1_pure_synthetic_summary.json'
STATION_META = ROOT / 'outputs_external_bologna_wells' / 'processed' / 'station_metadata_combined.csv'
BOLOGNA_WELLS = ROOT / 'outputs_external_bologna_wells' / 'processed' / 'bologna_wells_long.csv'
ALL_WELLS = ROOT / 'outputs_external_bologna_wells' / 'processed' / 'all_wells_long.csv'
WELL_OVERVIEW = ROOT / 'outputs_well_validation' / 'well_groundwater_validation_overview.json'
WELL_BY_DEPTH = ROOT / 'outputs_well_validation' / 'well_groundwater_validation_by_depth.csv'
WELL_BY_GWB = ROOT / 'outputs_well_validation' / 'well_groundwater_validation_by_gwb.csv'
WELL_TRUSTED_GWB = ROOT / 'outputs_well_validation' / 'well_groundwater_validation_trusted_gwb.csv'
WELL_TRUSTED_STATIONS = ROOT / 'outputs_well_validation' / 'well_groundwater_validation_trusted_stations.csv'
SWOT_SUMMARY = ROOT / 'outputs_external_constraints' / 'swot_bologna_overlap2025' / 'swot_bologna_overlap_summary.json'
FIG_DIR = ROOT / 'outputs_well_validation' / 'figures'

with open(OVERLAP_SUMMARY) as f:
    overlap = json.load(f)
with open(TILE_SUMMARY) as f:
    tiled_summary = json.load(f)
with open(REGIONAL_SUMMARY) as f:
    regional = json.load(f)
with open(SYNTH_SUMMARY) as f:
    synth = json.load(f)
with open(WELL_OVERVIEW) as f:
    well_overview = json.load(f)
with open(SWOT_SUMMARY) as f:
    swot_summary = json.load(f)

tiled = np.load(TILE_RESULTS)
regional_npz = np.load(REGIONAL_RESULTS)
stations = pd.read_csv(STATION_META)
bologna_wells = pd.read_csv(BOLOGNA_WELLS, parse_dates=['date'])
all_wells = pd.read_csv(ALL_WELLS, parse_dates=['date'])
well_by_depth = pd.read_csv(WELL_BY_DEPTH)
well_by_gwb = pd.read_csv(WELL_BY_GWB)
well_trusted_gwb = pd.read_csv(WELL_TRUSTED_GWB)
well_trusted_stations = pd.read_csv(WELL_TRUSTED_STATIONS)

state_names = [str(s) for s in tiled['state_names'].tolist()]
import ipywidgets as widgets


In [ ]:
# ── helper functions for interactive maps ─────────────────────────────────────
def robust_limits(a, pct=99):
    vals = np.asarray(a)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return (-1.0, 1.0)
    lim = float(np.nanpercentile(np.abs(vals), pct))
    lim = max(lim, 1e-6)
    return (-lim, lim)

def linear_trend_map(arr_tyx):
    t = np.arange(arr_tyx.shape[0], dtype=np.float32) - arr_tyx.shape[0] / 2.0
    denom = float(np.sum(t ** 2))
    flat = arr_tyx.reshape(arr_tyx.shape[0], -1).astype(float)
    slopes = (t[:, None] * flat).sum(axis=0) / denom
    return slopes.reshape(arr_tyx.shape[1:])

def plot_lonlat(ax, lon, lat, field, title, cmap='RdBu_r', symmetric=True):
    if symmetric:
        vmin, vmax = robust_limits(field)
    else:
        vals = np.asarray(field)[np.isfinite(field)]
        vmin = float(np.nanpercentile(vals, 1)) if vals.size else 0.0
        vmax = float(np.nanpercentile(vals, 99)) if vals.size else 1.0
    mesh = ax.pcolormesh(lon, lat, field, shading='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.colorbar(mesh, ax=ax, shrink=0.8, label='mm')
    return mesh


## Take-Home Message

- The validated real-data result is the **grouped Stage 1** multisensor model.
- It uses **InSAR + GRACE + refreshed SMAP** over the corrected Bologna overlap.
- The grouped `Groundwater` signal has **independent support from wells**.
- SWOT was prepared and tested, but is not a major driver of the current best result because the usable date overlap is sparse.
- Full layered hydrology is **not** the validated result yet.


## Paper Figure And Table Coverage

This section checks the requested paper items against what is now available in this notebook.


In [ ]:
coverage = pd.DataFrame([
    {'item': 'Figure 1. Study area + data domain + overlap', 'status': 'present', 'location': 'next section'},
    {'item': 'Figure 2. Method workflow schematic', 'status': 'present', 'location': 'next section'},
    {'item': 'Figure 3. Synthetic validation summary', 'status': 'present', 'location': 'next section'},
    {'item': 'Figure 4. Main real-data grouped Stage 1 result', 'status': 'present', 'location': 'grouped Stage 1 section'},
    {'item': 'Figure 5. Well validation figure', 'status': 'present', 'location': 'well validation section'},
    {'item': 'Table 1. Data summary', 'status': 'present', 'location': 'tables section'},
    {'item': 'Table 2. Synthetic and real experiment design summary', 'status': 'present', 'location': 'tables section'},
    {'item': 'Table 3. Main quantitative results', 'status': 'present', 'location': 'tables section'},
    {'item': 'Top-6 station time series', 'status': 'supplementary', 'location': 'figure gallery'},
    {'item': 'Lag histogram by state', 'status': 'supplementary', 'location': 'figure gallery'},
    {'item': 'Depth-class validation summary', 'status': 'supplementary', 'location': 'figure gallery'},
    {'item': 'Trusted aquifer groups', 'status': 'supplementary', 'location': 'figure gallery'},
    {'item': 'SWOT date-matching diagnostic', 'status': 'table/summary only', 'location': 'tables section'},
])
display(coverage)


## Figure 1. Study Area + Data Domain + Overlap

Study area, shared Bologna InSAR/W3RA overlap domain, native grouped inversion grid, and groundwater-well validation network.


In [ ]:
bbox = overlap['bbox']
fig = plt.figure(figsize=(12, 5), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[1.4, 1.0])
ax = fig.add_subplot(gs[0, 0])
inset = fig.add_subplot(gs[0, 1])

ax.scatter(stations['lon'], stations['lat'], s=8, c='0.80', label='Emilia-Romagna wells', alpha=0.6)
bo = stations.loc[stations['province'] == 'BO']
ax.scatter(bo['lon'], bo['lat'], s=12, c='#1f77b4', label='Bologna well stations', alpha=0.8)
rect = Rectangle((bbox['lon_min'], bbox['lat_min']), bbox['lon_max'] - bbox['lon_min'], bbox['lat_max'] - bbox['lat_min'],
                 fill=False, ec='crimson', lw=2.0, label='Shared Bologna overlap')
ax.add_patch(rect)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Emilia-Romagna wells and Bologna overlap domain')
ax.legend(loc='lower left', fontsize=8)

# simple depiction of the 22x24 shared grid
lon_edges = np.linspace(bbox['lon_min'], bbox['lon_max'], 25)
lat_edges = np.linspace(bbox['lat_min'], bbox['lat_max'], 23)
for x in lon_edges:
    ax.plot([x, x], [bbox['lat_min'], bbox['lat_max']], color='crimson', alpha=0.12, lw=0.7)
for y in lat_edges:
    ax.plot([bbox['lon_min'], bbox['lon_max']], [y, y], color='crimson', alpha=0.12, lw=0.7)
ax.text(bbox['lon_min'] + 0.05, bbox['lat_max'] - 0.08, '22 x 24 shared grid', color='crimson', fontsize=9, va='top')

inset.scatter(stations['lon'], stations['lat'], s=5, c='0.75', alpha=0.6)
inset.add_patch(Rectangle((bbox['lon_min'], bbox['lat_min']), bbox['lon_max'] - bbox['lon_min'], bbox['lat_max'] - bbox['lat_min'], fill=False, ec='crimson', lw=2.0))
inset.set_xlim(6.0, 14.5)
inset.set_ylim(41.5, 47.5)
inset.set_xlabel('Longitude')
inset.set_ylabel('Latitude')
inset.set_title('Northern Italy context (schematic)')

plt.show()


## Figure 2. Method Workflow Schematic

Full study workflow: synthetic validation of the deformation-space inversion machinery, followed by grouped multisensor Stage 1 inversion on Bologna and independent well validation.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')

boxes = [
    (0.03, 0.62, 0.18, 0.24, 'Synthetic pre-stage\n5-layer W3RA-like states\n[S0, Ss, Sd, Sg, Sr]'),
    (0.28, 0.62, 0.18, 0.24, 'Forward physics\nDeformation-space Z\nSynthetic Y'),
    (0.53, 0.62, 0.18, 0.24, 'Synthetic Stage 1\nMCMC validation\nState/deformation skill'),
    (0.78, 0.62, 0.18, 0.24, 'Optional synthetic Stage 2\nResidual / lag check\nExploratory only'),
    (0.03, 0.16, 0.18, 0.24, 'Real InSAR\nMintPy anomalies\nBologna overlap'),
    (0.28, 0.16, 0.18, 0.24, 'W3RA grouped prior\n[S0+Ss, Sd+Sr, Sg]\nShared 22x24 grid'),
    (0.53, 0.16, 0.18, 0.24, 'External constraints\nGRACE + SMAP\n+ exploratory SWOT'),
    (0.78, 0.16, 0.18, 0.24, 'Grouped Stage 1 posterior\nThen well validation\nMain supported result'),
]
for x, y, w, h, txt in boxes:
    patch = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02', fc='#f7f7f7', ec='#4c4c4c', lw=1.5)
    ax.add_patch(patch)
    ax.text(x + w/2, y + h/2, txt, ha='center', va='center', fontsize=10)

arrows = [
    ((0.21, 0.74), (0.28, 0.74)),
    ((0.46, 0.74), (0.53, 0.74)),
    ((0.71, 0.74), (0.78, 0.74)),
    ((0.21, 0.28), (0.28, 0.28)),
    ((0.46, 0.28), (0.53, 0.28)),
    ((0.71, 0.28), (0.78, 0.28)),
]
for (x0, y0), (x1, y1) in arrows:
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle='->', lw=2))

ax.text(0.5, 0.50, 'Synthetic pre-stage validates the machinery; real-data Stage 1 becomes grouped and is externally tested with wells.', ha='center', va='center', fontsize=11, fontweight='bold')
plt.show()


## Figure 3. Synthetic Validation Summary

The synthetic stage is included to show that the inversion machinery works under model-consistent conditions, but it is not the final applied result.


In [ ]:
synth_rows = [
    {'metric': 'Sg state $R^2$', 'value': synth['state_metrics']['Sg']['r2']},
    {'metric': 'Load total $R^2$', 'value': synth['derived_state_metrics']['Load_total']['r2']},
    {'metric': 'TWS $R^2$', 'value': synth['derived_state_metrics']['TWS']['r2']},
    {'metric': 'Deformation $R^2$', 'value': synth['deformation_metrics']['r2']},
]
synth_df = pd.DataFrame(synth_rows)
display(synth_df.round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(synth_df['metric'], synth_df['value'], color=['#1b9e77', '#d95f02', '#7570b3', '#4c78a8'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('$R^2$')
ax.set_title('Synthetic Stage 1 skill in the model-consistent case')
ax.tick_params(axis='x', rotation=20)
plt.show()


## Figure 4 And Tables 1-3

This section contains the main real-data grouped Stage 1 result together with the three main paper tables.


In [ ]:
table1 = pd.DataFrame([
    ['InSAR', 'MintPy time series', '2017-01-04 to 2025-06-27, 394 acquisitions, 555 interferograms', 'LOS deformation', 'Main observation stream', 'MintPy inversion; anomalies; aggregated to W3RA grid'],
    ['W3RA', 'Local W3RA overlap products', '2017-01-04 to 2024-08-01, shared 22x24 grid', 'S0, Ss, Sd, Sg, Sr', 'Model prior / grouped state basis', 'Overlap construction and anomaly conversion'],
    ['GRACE', 'JPL mascon RL06.3', '362 aligned overlap dates', 'lwe_thickness anomaly', 'Regional TWS-like constraint', 'Regional mean over overlap bbox'],
    ['SMAP', 'SPL3SMP_E', '358 downloads, 254 valid dates', 'surface soil moisture', 'Shallow hydrologic constraint', 'Regional mean over overlap bbox'],
    ['SWOT', 'RiverSP + LakeSP', 'Sparse same-window overlap; 9 useful matched dates after nearest-date fusion', 'surface-water summaries', 'Exploratory surface-water constraint', 'BBox subset and nearest-date matching'],
    ['Wells', 'ARPAE manual + automatic', '2009-01-01 to 2024-12-18', 'piezometric head, depth to water', 'Independent validation', 'Metadata merge, lon/lat conversion, lagged anomaly comparison'],
], columns=['Dataset', 'Product / source', 'Coverage used', 'Variables used', 'Role in study', 'Preprocessing before assimilation'])

table2 = pd.DataFrame([
    ['Synthetic Stage 1', '[S0, Ss, Sd, Sg, Sr]', 'Synthetic deformation Y and model-side Z', 'Validate inversion machinery', 'Posterior synthetic state and deformation skill'],
    ['Synthetic Stage 2', 'Residual / lag correction on synthetic posterior', 'Synthetic InSAR window + Stage 1 prior', 'Check whether learned residuals add value', 'Exploratory synthetic residual results'],
    ['Real grouped Stage 1', '[S0+Ss, Sd+Sr, Sg]', 'InSAR + GRACE + refreshed SMAP', 'Stable applied estimator', 'Grouped posterior state'],
    ['Exploratory SWOT extension', '[S0+Ss, Sd+Sr, Sg]', 'InSAR + GRACE + SMAP + SWOT', 'Test added surface-water information', 'No material improvement under sparse overlap'],
    ['Well validation', 'Grouped posterior vs well anomalies', 'ARPAE manual + automatic wells', 'Independent support for groundwater signal', 'Station-wise lagged correlations and trusted subset'],
], columns=['Stage', 'State formulation', 'Observations', 'Goal', 'Output'])

table3 = pd.DataFrame([
    ['Regional InSAR posterior $R^2$', regional['metrics']['insar_post']['r2']],
    ['Regional GRACE posterior $R^2$', regional['metrics']['grace_post']['r2']],
    ['Regional SMAP posterior $R^2$', regional['metrics']['smap_post']['r2']],
    ['Tiled InSAR posterior $R^2$', tiled_summary['metrics']['tile_insar_post']['r2']],
    ['max |ShallowLoad| (mm)', np.abs(tiled['x_tiles'][:, :, :, 0]).max()],
    ['max |DeepLoad| (mm)', np.abs(tiled['x_tiles'][:, :, :, 1]).max()],
    ['max |Groundwater| (mm)', np.abs(tiled['x_tiles'][:, :, :, 2]).max()],
    ['Well series evaluated', well_overview['n_station_series_evaluated']],
    ['Median well correlation', well_overview['median_corr']],
    ['Wells with corr >= 0.3', well_overview['n_corr_ge_0_3']],
    ['Wells with corr >= 0.5', well_overview['n_corr_ge_0_5']],
    ['Trusted groups / stations', f"{well_overview['n_trusted_gwb_groups']} / {well_overview['n_trusted_stations']}"],
], columns=['Metric', 'Value'])

print('Table 1. Data summary')
display(table1)
print('Table 2. Synthetic and real experiment design summary')
display(table2)
print('Table 3. Main quantitative results')
display(table3.round(4))

fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
last_idx = -1
for j, name in enumerate(state_names):
    im = axes[0, j].pcolormesh(tiled['lon_tiles'], tiled['lat_tiles'], tiled['x_tiles'][last_idx, :, :, j], shading='auto', cmap='RdBu_r')
    axes[0, j].set_title(name)
    axes[0, j].set_xlabel('Longitude')
    axes[0, j].set_ylabel('Latitude')
    plt.colorbar(im, ax=axes[0, j], shrink=0.8)

for ax, obs_key, prior_key, post_key, title in [
    (axes[1,0], 'y_insar', 'y_insar_prior', 'y_insar_post', 'InSAR'),
    (axes[1,1], 'y_grace', 'y_grace_prior', 'y_grace_post', 'GRACE'),
    (axes[1,2], 'y_smap', 'y_smap_prior', 'y_smap_post', 'SMAP'),
]:
    t = pd.to_datetime(regional_npz['time'])
    ax.plot(t, regional_npz[obs_key], label='Observed', lw=1.6)
    ax.plot(t, regional_npz[prior_key], label='Prior', lw=1.1, alpha=0.8)
    ax.plot(t, regional_npz[post_key], label='Posterior', lw=1.6)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=25)
axes[1,0].legend(loc='best', fontsize=8)
plt.show()


## Interactive Grouped State Map Viewer

Use the **Field** dropdown and **Time** slider to explore the grouped Kalman posterior
spatial maps for ShallowLoad, DeepLoad, and Groundwater across the Bologna overlap domain.


In [ ]:
_state_names = [str(s) for s in tiled['state_names'].tolist()]
_lon_tiles   = tiled['lon_tiles']
_lat_tiles   = tiled['lat_tiles']
_T           = tiled['x_tiles'].shape[0]
_times_pd    = pd.to_datetime(regional_npz['time'])

def _get_tile_field(field, tidx):
    if field in _state_names:
        return tiled['x_tiles'][tidx, :, :, _state_names.index(field)]
    if field == 'insar_obs':
        return tiled['y_obs_tiles'][tidx] * 1000   # m → mm
    if field == 'insar_post':
        return tiled['y_post_tiles'][tidx] * 1000
    raise ValueError(field)

_field_dd = widgets.Dropdown(
    options=_state_names + ['insar_obs', 'insar_post'],
    value='Groundwater',
    description='Field:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px'),
)
_time_sl = widgets.IntSlider(
    value=0, min=0, max=_T - 1, step=1,
    description='Time index:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px'),
)
_date_lbl = widgets.Label(value=str(_times_pd[0].date()))
_map_out   = widgets.Output()

def _refresh_map(*_):
    tidx = _time_sl.value
    _date_lbl.value = str(_times_pd[tidx].date())
    with _map_out:
        _map_out.clear_output(wait=True)
        arr = _get_tile_field(_field_dd.value, tidx)
        fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
        plot_lonlat(ax, _lon_tiles, _lat_tiles, arr,
                    f'{_field_dd.value}  |  {_times_pd[tidx].strftime("%Y-%m-%d")}')
        plt.show()

_field_dd.observe(_refresh_map, names='value')
_time_sl.observe(_refresh_map, names='value')
display(widgets.HBox([_field_dd, _date_lbl]), _time_sl, _map_out)
_refresh_map()


## Interactive Click-on-Map Timeseries Viewer

Click any grid cell on the InSAR velocity map below to plot the full 2017–2024
deformation timeseries for that pixel alongside the grouped posterior state.


In [ ]:
import xarray as xr

_insar_ds  = xr.open_dataset(
    ROOT / 'outputs_bologna_2025_overlap' / 'insar_mintpy2025_on_w3ra_grid.nc'
)
_defo_mm   = _insar_ds['insar_deformation'].values.astype(float) * 1000.0  # m → mm
_insar_lon = _insar_ds['lon'].values
_insar_lat = _insar_ds['lat'].values
_insar_t   = pd.to_datetime(_insar_ds.time.values)
_insar_ds.close()

_vel_map = linear_trend_map(_defo_mm) * 365.25   # mm/acquisition → mm/yr approx

_click_fig, _click_ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
plot_lonlat(_click_ax, _insar_lon, _insar_lat, _vel_map,
            'InSAR LOS velocity (mm/yr) — click a pixel', cmap='RdBu_r')
_click_out = widgets.Output()

def _nearest_ij(xlon, ylat, lon2d, lat2d):
    dist2 = (lon2d - xlon) ** 2 + (lat2d - ylat) ** 2
    return np.unravel_index(np.nanargmin(dist2), dist2.shape)

def _on_map_click(event):
    if event.inaxes is not _click_ax or event.xdata is None:
        return
    row, col = _nearest_ij(event.xdata, event.ydata, _insar_lon, _insar_lat)
    ts = _defo_mm[:, row, col]
    lon_pt = float(_insar_lon[row, col])
    lat_pt = float(_insar_lat[row, col])
    with _click_out:
        _click_out.clear_output(wait=True)
        fig2, axes2 = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
        # left: deformation timeseries
        axes2[0].plot(_insar_t, ts, lw=1.2, color='#1f77b4')
        axes2[0].set_title(f'InSAR LOS  ({lat_pt:.2f}°N, {lon_pt:.2f}°E)', fontsize=10)
        axes2[0].set_ylabel('Cumulative LOS (mm)')
        axes2[0].tick_params(axis='x', rotation=30)
        # right: grouped states at same pixel
        state_colors = ['#e6550d', '#31a354', '#756bb1']
        for si, sn in enumerate(_state_names):
            sv = tiled['x_tiles'][:, row, col, si]
            axes2[1].plot(_times_pd, sv, lw=1.2, color=state_colors[si], label=sn)
        axes2[1].set_title(f'Grouped states  ({lat_pt:.2f}°N, {lon_pt:.2f}°E)', fontsize=10)
        axes2[1].set_ylabel('State anomaly (mm)')
        axes2[1].tick_params(axis='x', rotation=30)
        axes2[1].legend(fontsize=8)
        plt.show()

_click_fig.canvas.mpl_connect('button_press_event', _on_map_click)
display(_click_out)


## Figure 5. Well Validation

Independent validation of the grouped posterior against Bologna groundwater wells, including the conservative trusted subset used for interpretation.


In [ ]:
print('Overall well validation summary')
display(pd.DataFrame([well_overview]).round(4))
print('Validation by depth class')
display(well_by_depth.round(4))
print('Trusted hydrogeologic groups')
display(well_trusted_gwb.round(4))
print('Trusted stations (first 21 rows)')
display(well_trusted_stations[['station_code','municipality','gwb_name','depth_class','best_state','corr_anom','best_lag_days']].head(21).round(4))

for name in ['well_validation_summary_panel.png', 'trusted_wells_map.png']:
    path = FIG_DIR / name
    print(path)
    display(IPyImage(filename=str(path)))


## Supplementary Figures And Tables

These are strong supporting items that do not all need to sit in the main paper.


In [ ]:
supp_df = pd.DataFrame([
    {'item': 'Top 6 station time series', 'recommended_place': 'Supplementary'},
    {'item': 'Lag histogram by state', 'recommended_place': 'Supplementary'},
    {'item': 'Depth-class validation summary', 'recommended_place': 'Supplementary'},
    {'item': 'Trusted hydrogeologic groups', 'recommended_place': 'Supplementary'},
    {'item': f"SWOT overlap summary: {swot_summary['river_times']} river dates, {swot_summary['lake_times']} lake dates", 'recommended_place': 'Supplementary / text'},
])
display(supp_df)
for name in ['trusted_aquifer_groups.png', 'lag_histogram_by_state.png', 'depth_class_summary.png', 'top6_station_timeseries.png']:
    path = FIG_DIR / name
    print(path)
    display(IPyImage(filename=str(path)))
